In LoRA, we freeze a selected weight matrix \(W\) and train two smaller matrices:

$
W_{\text{adapted}}=W+\frac{\alpha}{r}BA
$

Here, \(r\) is the rank and $(alpha/r)$ scales the update.

# Domain Adaptation through LoRA

LoRA = Low-Rank Adaptation

Instead of training a complete weight matrix W, train two smaller
matrices A and B.

For a column-vector input x:

Original transformation:
y = Wx

LoRA transformation:
y = Wx + (alpha / rank) BAx

During training:
- Original model parameters remain frozen.
- Only the LoRA matrices A and B change.

After training, the correction can be merged into the original weight.

This notebook applies LoRA to the combined query/key/value
projection in each attention layer.

In [7]:
!ollama list

NAME                       ID              SIZE      MODIFIED     
qwen3:8b                   500a1f067a9f    5.2 GB    36 hours ago    
nomic-embed-text:latest    0a109f422b47    274 MB    2 days ago      
llama3.2:1b                baf6a787fdff    1.3 GB    2 days ago      
qwen2.5:1.5b               65ec06548149    986 MB    3 weeks ago     


In [6]:
%pip install -q torch "transformers>=4.45,<5" "peft>=0.13,<0.19" pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import hashlib
import math
import random

import pandas as pd
import torch

from IPython.display import display
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

# Choose "cpu", "cuda", or "auto".
DEVICE_CHOICE = "cpu"

if DEVICE_CHOICE == "auto":
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable. Choose "cpu".')
    device = torch.device("cuda")
elif DEVICE_CHOICE == "cpu":
    device = torch.device("cpu")
else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [8]:
# MODEL_NAME = "distilbert/distilgpt2"
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# Set the intermediate dimension of the low-rank update.
LORA_RANK = 4
# Control the strength of the LoRA update.
LORA_ALPHA = 8
# Apply no dropout in the LoRA branch.
LORA_DROPOUT = 0.0

MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 5e-4

print("LoRA rank:", LORA_RANK)
print("LoRA alpha:", LORA_ALPHA)
print("Scaling factor:", LORA_ALPHA / LORA_RANK)

LoRA rank: 4
LoRA alpha: 8
Scaling factor: 2.0


In [9]:
# Training and evaluation data

In [10]:
train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

assert set(train_texts).isdisjoint(eval_texts)

print("Training examples:", len(train_texts))
print("Evaluation examples:", len(eval_texts))

Training examples: 12
Evaluation examples: 4


In [11]:
# Loading a fresh model and prepare batches

In [12]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
).to(device)

base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.use_cache = False

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_NAME,
#     local_files_only=True,
# )
# tokenizer.pad_token = tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     local_files_only=True,
#     attn_implementation="eager",
# ).to(device)

# base_model.config.pad_token_id = tokenizer.pad_token_id
# base_model.config.use_cache = False


def collate_texts(texts):
    batch = tokenizer(
        [text + tokenizer.eos_token for text in texts],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100

    batch["labels"] = labels
    return batch


train_loader = DataLoader(
    train_texts,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_texts,
)

eval_loader = DataLoader(
    eval_texts,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_texts,
)

print("Model loaded.")

print("\nAttention projections selected for LoRA:")
for name, module in base_model.named_modules():
    if name.endswith("c_attn"):# LLAMA
        print(name, tuple(module.weight.shape))
    elif name.endswith(("q_proj", "v_proj")):# QWEN
    print(name, tuple(module.weight.shape))

tokenizer_config.json: 0.00B [00:00, ?B/s]

C:\Users\fedex\anaconda3\envs\ABMLLM\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\fedex\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded.

Attention projections selected for LoRA:


In [ ]:
# Define evaluation and completion functions

In [13]:
@torch.no_grad()
def evaluate_loss(current_model):
    current_model.eval()

    loss_sum = 0.0
    token_count = 0

    for batch in eval_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        output = current_model(**batch)

        valid_tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        loss_sum += output.loss.item() * valid_tokens
        token_count += valid_tokens

    return loss_sum / token_count


@torch.no_grad()
def complete_text(current_model, prompt):
    current_model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(device)

    output_ids = current_model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_length = inputs["input_ids"].shape[1]

    return tokenizer.decode(
        output_ids[0, prompt_length:],
        skip_special_tokens=True,
    ).strip()


test_prompts = [
    "Predictive maintenance is",
    "When bearing temperature increases,",
    "Vibration monitoring helps engineers",
]

In [14]:
# Recording the original baseline

In [15]:
original_loss = evaluate_loss(base_model)

original_completions = {
    prompt: complete_text(base_model, prompt)
    for prompt in test_prompts
}

print(f"Original evaluation loss: {original_loss:.4f}")
print(f"Original perplexity: {math.exp(original_loss):.2f}")

for prompt, answer in original_completions.items():
    print("\nPROMPT:", prompt)
    print("CONTINUATION:", answer)

Original evaluation loss: 4.0746
Original perplexity: 58.83

PROMPT: Predictive maintenance is
CONTINUATION: a proactive approach to equipment maintenance that aims to prevent equipment failure before it occurs. It involves using data analytics and machine learning algorithms to predict when equipment will fail based on its performance and usage patterns. Predictive maintenance can help organizations

PROMPT: When bearing temperature increases,
CONTINUATION: the viscosity of the lubricating oil ____.
A. Increases
B. Decreases
C. Remains unchanged
D. Cannot be determined
Answer:
B

The main reason for the increase in bearing temperature is ____

PROMPT: Vibration monitoring helps engineers
CONTINUATION: to identify and prevent equipment failures. It is a critical part of the maintenance process and can help to reduce downtime and increase productivity. Vibration monitoring can be used to detect a wide range of issues, including bearing failures, loose


In [ ]:
# Attach LoRA matrices
# bias="none"  keeps all original biases frozen too.

In [18]:
# lora_config = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=LORA_RANK,
#     lora_alpha=LORA_ALPHA,
#     lora_dropout=LORA_DROPOUT,
#     target_modules=["c_attn"],
#     fan_in_fan_out=True,
#     bias="none",
#     init_lora_weights=True,
# )


# Configure LoRA for the Qwen causal language model.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # Train using next-token prediction.
    # Matrix A reduces the input to 4 dimensions.
    # Matrix B expands it to the projection's output dimension.
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,# Scale the correction by alpha / rank = 8 / 4 = 2.
    target_modules=["q_proj", "v_proj"], # Add LoRA to query and value projections in the attention layers.
    fan_in_fan_out=False,# Qwen uses standard Linear weight storage: output × input.
    bias="none", # Keep the original bias parameters frozen.
    # Use default initialization: A is random and B starts at zero.
    # Therefore, the initial LoRA correction B @ A is zero.
    init_lora_weights=True,
)

# Wraping the selected projections with LoRA branches.
# PEFT creates matrices A and B and freezes the original model parameters.
model = get_peft_model(base_model, lora_config)
model = model.to(device)# Move the model and its LoRA matrices to the selected CPU or GPU.

model.print_trainable_parameters()# Just displaying how many parameters are trainable.

# Verify that every trainable parameter belongs to LoRA.
assert all(
    "lora_" in name
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
)

display(pd.DataFrame([
    {
        "parameter": name,
        "shape": str(tuple(parameter.shape)),
        "elements": parameter.numel(),
    }
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]))

trainable params: 544,768 || all params: 1,544,259,072 || trainable%: 0.0353


,parameter,shape,elements
0,base_model.model.model.layers.0.self_attn.q_pr...,"(4, 1536)",6144
1,base_model.model.model.layers.0.self_attn.q_pr...,"(1536, 4)",6144
2,base_model.model.model.layers.0.self_attn.v_pr...,"(4, 1536)",6144
3,base_model.model.model.layers.0.self_attn.v_pr...,"(256, 4)",1024
4,base_model.model.model.layers.1.self_attn.q_pr...,"(4, 1536)",6144
...,...,...,...
107,base_model.model.model.layers.26.self_attn.v_p...,"(256, 4)",1024
108,base_model.model.model.layers.27.self_attn.q_p...,"(4, 1536)",6144
109,base_model.model.model.layers.27.self_attn.q_p...,"(1536, 4)",6144
110,base_model.model.model.layers.27.self_attn.v_p...,"(4, 1536)",6144


In [19]:
# Inspect matrices A and B

In [22]:


# Find the first projection that has a LoRA branch.
# This avoids relying on a model-specific layer path.

projection_name, first_projection = next(
    (name, module)
    for name, module in model.named_modules()
    if hasattr(module, "lora_A") and "default" in module.lora_A
)

# A maps the input dimension to the small rank dimension.
# Shape: (rank, input_dimension).
A = first_projection.lora_A["default"].weight
# B maps the rank dimension to the projection's output dimension.
# Shape: (output_dimension, rank).
B = first_projection.lora_B["default"].weight

print("Selected projection:", projection_name)
print("LoRA A shape:", tuple(A.shape))
print("LoRA B shape:", tuple(B.shape))


# SPECIFIC TO QWEN

# first_projection = (
#     model.base_model.model.model.layers[0].self_attn.q_proj
# )

# A = first_projection.lora_A["default"].weight
# B = first_projection.lora_B["default"].weight

# print("LoRA A shape:", tuple(A.shape))
# print("LoRA B shape:", tuple(B.shape))


## SPECIFIC TO LLAMA

# first_projection = model.base_model.model.transformer.h[0].attn.c_attn

# A = first_projection.lora_A["default"].weight
# B = first_projection.lora_B["default"].weight

# print("A shape:", tuple(A.shape))
# print("B shape:", tuple(B.shape))

# Compare the added parameter count with the original projection.
print("Parameters in A:", A.numel())
print("Parameters in B:", B.numel())

print(
    "Parameters in original projection:",
    first_projection.base_layer.weight.numel(),
)

# Displaying the multiplier applied to the LoRA correction.
print(
    "LoRA scaling:",
    first_projection.scaling["default"],
)

# B should initially contain zeros, so the initial correction is zero.
print(
    "Initial B magnitude:",
    B.detach().abs().max().item(),
)

Selected projection: base_model.model.model.layers.0.self_attn.q_proj
LoRA A shape: (4, 1536)
LoRA B shape: (1536, 4)
Parameters in A: 6144
Parameters in B: 6144
Parameters in original projection: 2359296
LoRA scaling: 2.0
Initial B magnitude: 0.0


The output dimension is 2304 because c_attn produces queries, keys, and values together: \(3\times768\).

For one projection:
$ 
  N_{\text{LoRA}}=r(d_{\text{in}}+d_{\text{out}}) =4(768+2304)=12,288
$ 

Across six layers, this gives 73,728 trainable parameters.
With the initialization used here, \(B=0\), so the initial LoRA correction is zero.

In [ ]:
# Verify initialization and record paramet

In [23]:
initialized_loss = evaluate_loss(model)

print(f"Original loss:         {original_loss:.6f}")
print(f"Initialized LoRA loss: {initialized_loss:.6f}")

assert math.isclose(
    original_loss,
    initialized_loss,
    rel_tol=1e-5,
    abs_tol=1e-5,
)


def frozen_fingerprints(current_model):
    return {
        name: hashlib.sha256(
            parameter.detach().cpu().contiguous().numpy().tobytes()
        ).hexdigest()
        for name, parameter in current_model.named_parameters()
        if not parameter.requires_grad
    }

# Save fingerprints of all frozen original parameter tensors.
frozen_before = frozen_fingerprints(model)


# Save independent copies of all trainable LoRA matrices.
# We will compare these values after training.
lora_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

print("Verified: initialized LoRA preserves the baseline.")

Original loss:         4.074642
Initialized LoRA loss: 4.074642
Verified: initialized LoRA preserves the baseline.


In [24]:
# Train only LoRA parameters

In [25]:
# Select only trainable parameters: the LoRA A and B matrices.

trainable_parameters = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]
# Give only these matrices to the optimizer.

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE,
    weight_decay=0.0,
)

training_history = []
iterator = iter(train_loader)

model.train()

for step in range(1, TRAINING_STEPS + 1):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(train_loader)
        batch = next(iterator)

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    # Inside the actual training loop:

    # Clear gradients from the previous step.
    optimizer.zero_grad(set_to_none=True)
    # Run the model and calculate next-token prediction loss.
    # Selected projections automatically compute:
    # original_output + scaling * B(A(input)).
    loss = model(**batch).loss
    # Calculate gradients for A and B through the model computation.
    # The original model weights remain frozen.
    loss.backward()
    # Limit gradient magnitude to help stabilize training.
    torch.nn.utils.clip_grad_norm_(
        trainable_parameters,
        max_norm=1.0,
    )

    optimizer.step()# Actually update the trainable A and B matrices.

    training_history.append({
        "step": step,
        "training_loss": loss.item(),
    })

    if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
        print(
            f"Step {step}/{TRAINING_STEPS} "
            f"| Loss: {loss.item():.4f}"
        )

print("\nLoRA training complete.")

Step 1/30 | Loss: 5.0803
Step 5/30 | Loss: 3.4685
Step 10/30 | Loss: 2.7866
Step 15/30 | Loss: 4.3089
Step 20/30 | Loss: 3.6455
Step 25/30 | Loss: 3.2780
Step 30/30 | Loss: 1.6236

LoRA training complete.


In [26]:
## Compare before and after

In [27]:
trained_loss = evaluate_loss(model)

display(pd.DataFrame([
    {
        "stage": "Before LoRA training",
        "evaluation_loss": original_loss,
        "perplexity": math.exp(original_loss),
    },
    {
        "stage": "After LoRA training",
        "evaluation_loss": trained_loss,
        "perplexity": math.exp(trained_loss),
    },
]))

print(
    "Evaluation loss reduction:",
    f"{original_loss - trained_loss:.4f}",
)

for prompt in test_prompts:
    print("\nPROMPT:", prompt)
    print("\nBEFORE:", original_completions[prompt])
    print("\nAFTER:", complete_text(model, prompt))
    print("=" * 70)

,stage,evaluation_loss,perplexity
0,Before LoRA training,4.074642,58.829387
1,After LoRA training,2.796264,16.383325


Evaluation loss reduction: 1.2784

PROMPT: Predictive maintenance is

BEFORE: a proactive approach to equipment maintenance that aims to prevent equipment failure before it occurs. It involves using data analytics and machine learning algorithms to predict when equipment will fail based on its performance and usage patterns. Predictive maintenance can help organizations

AFTER: a proactive approach to equipment maintenance that uses data analysis to predict when equipment will fail.

PROMPT: When bearing temperature increases,

BEFORE: the viscosity of the lubricating oil ____.
A. Increases
B. Decreases
C. Remains unchanged
D. Cannot be determined
Answer:
B

The main reason for the increase in bearing temperature is ____

AFTER: the bearing clearance should be reduced.
A. Correct
B. Incorrect
Answer:
B

The main function of the oil reservoir in a transformer is to prevent the oil level from dropping due to oil evaporation or oil leakage

PROMPT: Vibration monitoring helps engineers

BE

In [28]:
### Positive loss reduction indicates improvement on these held-out sentences. This small exercise does not establish broad domain expertise.

In [29]:
# Verify that only LoRA parameters changed

In [30]:
# Recalculate fingerprints of all frozen original parameters.

frozen_after = frozen_fingerprints(model)

# Find frozen tensors whose values changed.

changed_frozen = [
    name
    for name in frozen_before
    if frozen_before[name] != frozen_after[name]
]

change_rows = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        after = parameter.detach().cpu()
        difference = after - lora_before[name]

        change_rows.append({
            "parameter": name,
            "changed": not torch.equal(after, lora_before[name]),
            "mean_absolute_change": difference.abs().mean().item(),
        })

change_df = pd.DataFrame(change_rows)

display(change_df)
print("Changed frozen tensors:", len(changed_frozen))


# Confirming that at least one LoRA parameter tensor changed.
assert change_df["changed"].any()
assert len(changed_frozen) == 0 # Confirming that none of the frozen original parameter tensors changed.

print("Verified: LoRA parameters changed; original parameters stayed identical.")

,parameter,changed,mean_absolute_change
0,base_model.model.model.layers.0.self_attn.q_pr...,True,0.002995
1,base_model.model.model.layers.0.self_attn.q_pr...,True,0.002923
2,base_model.model.model.layers.0.self_attn.v_pr...,True,0.003655
3,base_model.model.model.layers.0.self_attn.v_pr...,True,0.003460
4,base_model.model.model.layers.1.self_attn.q_pr...,True,0.003658
...,...,...,...
107,base_model.model.model.layers.26.self_attn.v_p...,True,0.004582
108,base_model.model.model.layers.27.self_attn.q_p...,True,0.004118
109,base_model.model.model.layers.27.self_attn.q_p...,True,0.003784
110,base_model.model.model.layers.27.self_attn.v_p...,True,0.005014


Changed frozen tensors: 0
Verified: LoRA parameters changed; original parameters stayed identical.


In [31]:
# Optional: Merge LoRA into the model
# LoRA’s update can be merged into the base weights using merge_and_unload(). 
# The resulting model no longer needs separate LoRA branches

In [32]:
##  Run this after finishing training and verification. It changes the model structure.

In [33]:
model.eval()

probe_inputs = tokenizer(
    "Vibration monitoring helps engineers",
    return_tensors="pt",
).to(device)

with torch.no_grad():
    logits_before_merge = model(**probe_inputs).logits.detach().clone()

# Run only after training and the frozen-weight verification.
# Add the learned correction into each selected original weight:
# W_merged = W_original + (alpha / rank) * (B @ A).
# Remove the separate LoRA branches from the returned model.
merged_model = model.merge_and_unload(safe_merge=True)
merged_model.eval() # Set the merged model to evaluation mode.

with torch.no_grad():
    logits_after_merge = merged_model(**probe_inputs).logits

max_difference = (
    logits_before_merge - logits_after_merge
).abs().max().item()

print("Maximum logit difference after merging:", max_difference)

torch.testing.assert_close(
    logits_before_merge,
    logits_after_merge,
    rtol=1e-4,
    atol=1e-4,
)

print("Verified: merging preserves predictions within numerical tolerance.")

print(
    "Remaining LoRA parameter tensors:",
    sum(
        "lora_" in name
        for name, _ in merged_model.named_parameters()
    ),
)

Maximum logit difference after merging: 1.9550323486328125e-05
Verified: merging preserves predictions within numerical tolerance.
Remaining LoRA parameter tensors: 0


In [ ]:
# Exercises and takeaway

In [52]:
## Participant exercises

1. Change LORA_RANK from 4 to 8.
   Predict the trainable parameter count.

2. Set LORA_ALPHA to twice the rank.
   What happens to the scaling factor?

3. Compare 30 and 60 training steps.

4. Why does setting B to zero preserve the original model initially?

5. How does LoRA differ from a nonlinear bottleneck adapter?

6. Why can LoRA be merged into a weight matrix?

For a fresh experiment:
Change settings in Cell 4, then rerun Cells 6–14.

Run the optional merge cell only after finishing the experiment.

## Adapter tuning versus LoRA

Bottleneck adapter:
Adds a nonlinear transformation of hidden states.

LoRA:
Adds a low-rank linear correction to a selected weight transformation.

## Main takeaway

LoRA learns compact weight updates while keeping the original
model parameters frozen during training.

SyntaxError: invalid character '–' (U+2013) (1044947104.py, line 18)